# QCNN Image Dataset — PyTorch Version

PyTorch alternative to `QCNN_image_dataset.ipynb`.

**What changed:**
- Data loading & image resize → `torchvision` + `torch.nn.functional`
- Autoencoder dimensionality reduction → PyTorch `nn.Module`
- Training loop → `torch.optim` + `nn.CrossEntropyLoss` / `nn.MSELoss`
- Quantum circuit → PennyLane QNode with `interface="torch"`

Quantum ansatzes, embeddings, and QCNN architecture are identical to the original notebook.

## Dependencies

In [1]:
# pip install torch torchvision pennylane scikit-learn numpy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.decomposition import PCA
from torchvision import datasets

# Data
---

In [2]:
pca32 = ['pca32-1', 'pca32-2', 'pca32-3', 'pca32-4']
autoencoder32 = ['autoencoder32-1', 'autoencoder32-2', 'autoencoder32-3', 'autoencoder32-4']
pca30 = ['pca30-1', 'pca30-2', 'pca30-3', 'pca30-4']
autoencoder30 = ['autoencoder30-1', 'autoencoder30-2', 'autoencoder30-3', 'autoencoder30-4']
pca16 = ['pca16-1', 'pca16-2', 'pca16-3', 'pca16-4', 'pca16-compact']
autoencoder16 = ['autoencoder16-1', 'autoencoder16-2', 'autoencoder16-3', 'autoencoder16-4', 'autoencoder16-compact']
pca12 = ['pca12-1', 'pca12-2', 'pca12-3', 'pca12-4']
autoencoder12 = ['autoencoder12-1', 'autoencoder12-2', 'autoencoder12-3', 'autoencoder12-4']


def _load_raw_dataset(dataset):
    if dataset == 'fashion_mnist':
        train_ds = datasets.FashionMNIST(root='./data', train=True, download=True)
        test_ds = datasets.FashionMNIST(root='./data', train=False, download=True)
    elif dataset == 'mnist':
        train_ds = datasets.MNIST(root='./data', train=True, download=True)
        test_ds = datasets.MNIST(root='./data', train=False, download=True)
    else:
        raise ValueError(f"Unknown dataset: {dataset}")

    x_train = train_ds.data.numpy().astype(np.float32) / 255.0
    y_train = train_ds.targets.numpy()
    x_test = test_ds.data.numpy().astype(np.float32) / 255.0
    y_test = test_ds.targets.numpy()
    return x_train, y_train, x_test, y_test


def _resize_flat(X, height, width):
    """Resize (N, 28, 28) images to (N, height * width) using PyTorch."""
    t = torch.tensor(X, dtype=torch.float32).unsqueeze(1)  # (N, 1, 28, 28)
    t = F.interpolate(t, size=(height, width), mode='bilinear', align_corners=False)
    t = t.squeeze(1)
    if width == 1:
        t = t.squeeze(-1)
    return t.numpy().reshape(len(X), -1)


class Autoencoder(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, latent_dim),
            nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 784),
            nn.Sigmoid(),
        )

    def forward(self, x):
        flat = x.view(x.size(0), -1)
        encoded = self.encoder(flat)
        decoded = self.decoder(encoded)
        return decoded, encoded


def _train_autoencoder(X_train, X_test, latent_dim, epochs=10):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = Autoencoder(latent_dim).to(device)
    optimizer = optim.Adam(model.parameters())
    criterion = nn.MSELoss()

    X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
    X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)

    for _ in range(epochs):
        model.train()
        optimizer.zero_grad()
        recon, _ = model(X_train_t)
        loss = criterion(recon, X_train_t.view(X_train_t.size(0), -1))
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        _, train_enc = model(X_train_t)
        _, test_enc = model(X_test_t)
    return train_enc.cpu().numpy(), test_enc.cpu().numpy()


def data_load_and_process(dataset, classes=[0, 1], feature_reduction='resize256', binary=True):
    x_train, y_train, x_test, y_test = _load_raw_dataset(dataset)

    print(f"Loaded X_train shape: {x_train.shape}, Y_train shape: {y_train.shape}")
    print(f"Loaded X_test shape: {x_test.shape}, Y_test shape: {y_test.shape}")

    if classes == 'odd_even':
        odd = [1, 3, 5, 7, 9]
        X_train, X_test = x_train, x_test
        if binary:
            Y_train = np.array([1 if y in odd else -1 for y in y_train])
            Y_test = np.array([1 if y in odd else -1 for y in y_test])
        else:
            Y_train = np.array([1 if y in odd else 0 for y in y_train])
            Y_test = np.array([1 if y in odd else 0 for y in y_test])

    elif classes == '>4':
        greater = [5, 6, 7, 8, 9]
        X_train, X_test = x_train, x_test
        if binary:
            Y_train = np.array([1 if y in greater else -1 for y in y_train])
            Y_test = np.array([1 if y in greater else -1 for y in y_test])
        else:
            Y_train = np.array([1 if y in greater else 0 for y in y_train])
            Y_test = np.array([1 if y in greater else 0 for y in y_test])

    else:
        train_mask = (y_train == classes[0]) | (y_train == classes[1])
        test_mask = (y_test == classes[0]) | (y_test == classes[1])
        X_train, X_test = x_train[train_mask], x_test[test_mask]
        Y_train, Y_test = y_train[train_mask], y_test[test_mask]

        if binary:
            Y_train = np.array([1 if y == classes[0] else -1 for y in Y_train])
            Y_test = np.array([1 if y == classes[0] else -1 for y in Y_test])
        else:
            Y_train = np.array([1 if y == classes[0] else 0 for y in Y_train])
            Y_test = np.array([1 if y == classes[0] else 0 for y in Y_test])

    if feature_reduction == 'resize256':
        X_train = _resize_flat(X_train, 256, 1)
        X_test = _resize_flat(X_test, 256, 1)
        return X_train, X_test, Y_train, Y_test

    if feature_reduction == 'pca8' or feature_reduction in pca32 \
            or feature_reduction in pca30 or feature_reduction in pca16 or feature_reduction in pca12:
        X_train = _resize_flat(X_train, 784, 1)
        X_test = _resize_flat(X_test, 784, 1)

        if feature_reduction == 'pca8':
            pca = PCA(8)
        elif feature_reduction in pca32:
            pca = PCA(32)
        elif feature_reduction in pca30:
            pca = PCA(30)
        elif feature_reduction in pca16:
            pca = PCA(16)
        elif feature_reduction in pca12:
            pca = PCA(12)

        X_train = pca.fit_transform(X_train)
        X_test = pca.transform(X_test)

        if feature_reduction == 'pca8' or feature_reduction == 'pca16-compact' or \
                feature_reduction in pca30 or feature_reduction in pca12:
            X_train = (X_train - X_train.min()) * (np.pi / (X_train.max() - X_train.min()))
            X_test = (X_test - X_test.min()) * (np.pi / (X_test.max() - X_test.min()))
        return X_train, X_test, Y_train, Y_test

    if feature_reduction == 'autoencoder8' or feature_reduction in autoencoder32 \
            or feature_reduction in autoencoder30 or feature_reduction in autoencoder16 or feature_reduction in autoencoder12:
        if feature_reduction == 'autoencoder8':
            latent_dim = 8
        elif feature_reduction in autoencoder32:
            latent_dim = 32
        elif feature_reduction in autoencoder30:
            latent_dim = 30
        elif feature_reduction in autoencoder16:
            latent_dim = 16
        elif feature_reduction in autoencoder12:
            latent_dim = 12

        X_train, X_test = _train_autoencoder(X_train, X_test, latent_dim)

        if feature_reduction == 'autoencoder8' or feature_reduction == 'autoencoder16-compact' or \
                feature_reduction in autoencoder30 or feature_reduction in autoencoder12:
            X_train = (X_train - X_train.min()) * (np.pi / (X_train.max() - X_train.min()))
            X_test = (X_test - X_test.min()) * (np.pi / (X_test.max() - X_test.min()))
        return X_train, X_test, Y_train, Y_test

    raise ValueError(f"Unknown feature_reduction: {feature_reduction}")

# Unitary Ansätze
---

In [3]:
import pennylane as qml


def U_TTN(params, wires):
    qml.RY(params[0], wires=wires[0])
    qml.RY(params[1], wires=wires[1])
    qml.CNOT(wires=[wires[0], wires[1]])


def U_5(params, wires):
    qml.RX(params[0], wires=wires[0])
    qml.RX(params[1], wires=wires[1])
    qml.RZ(params[2], wires=wires[0])
    qml.RZ(params[3], wires=wires[1])
    qml.CRZ(params[4], wires=[wires[1], wires[0]])
    qml.CRZ(params[5], wires=[wires[0], wires[1]])
    qml.RX(params[6], wires=wires[0])
    qml.RX(params[7], wires=wires[1])
    qml.RZ(params[8], wires=wires[0])
    qml.RZ(params[9], wires=wires[1])


def U_6(params, wires):
    qml.RX(params[0], wires=wires[0])
    qml.RX(params[1], wires=wires[1])
    qml.RZ(params[2], wires=wires[0])
    qml.RZ(params[3], wires=wires[1])
    qml.CRX(params[4], wires=[wires[1], wires[0]])
    qml.CRX(params[5], wires=[wires[0], wires[1]])
    qml.RX(params[6], wires=wires[0])
    qml.RX(params[7], wires=wires[1])
    qml.RZ(params[8], wires=wires[0])
    qml.RZ(params[9], wires=wires[1])


def U_9(params, wires):
    qml.Hadamard(wires=wires[0])
    qml.Hadamard(wires=wires[1])
    qml.CZ(wires=[wires[0], wires[1]])
    qml.RX(params[0], wires=wires[0])
    qml.RX(params[1], wires=wires[1])


def U_13(params, wires):
    qml.RY(params[0], wires=wires[0])
    qml.RY(params[1], wires=wires[1])
    qml.CRZ(params[2], wires=[wires[1], wires[0]])
    qml.RY(params[3], wires=wires[0])
    qml.RY(params[4], wires=wires[1])
    qml.CRZ(params[5], wires=[wires[0], wires[1]])


def U_14(params, wires):
    qml.RY(params[0], wires=wires[0])
    qml.RY(params[1], wires=wires[1])
    qml.CRX(params[2], wires=[wires[1], wires[0]])
    qml.RY(params[3], wires=wires[0])
    qml.RY(params[4], wires=wires[1])
    qml.CRX(params[5], wires=[wires[0], wires[1]])


def U_15(params, wires):
    qml.RY(params[0], wires=wires[0])
    qml.RY(params[1], wires=wires[1])
    qml.CNOT(wires=[wires[1], wires[0]])
    qml.RY(params[2], wires=wires[0])
    qml.RY(params[3], wires=wires[1])
    qml.CNOT(wires=[wires[0], wires[1]])


def U_SO4(params, wires):
    qml.RY(params[0], wires=wires[0])
    qml.RY(params[1], wires=wires[1])
    qml.CNOT(wires=[wires[0], wires[1]])
    qml.RY(params[2], wires=wires[0])
    qml.RY(params[3], wires=wires[1])
    qml.CNOT(wires=[wires[0], wires[1]])
    qml.RY(params[4], wires=wires[0])
    qml.RY(params[5], wires=wires[1])


def U_SU4(params, wires):
    qml.U3(params[0], params[1], params[2], wires=wires[0])
    qml.U3(params[3], params[4], params[5], wires=wires[1])
    qml.CNOT(wires=[wires[0], wires[1]])
    qml.RY(params[6], wires=wires[0])
    qml.RZ(params[7], wires=wires[1])
    qml.CNOT(wires=[wires[1], wires[0]])
    qml.RY(params[8], wires=wires[0])
    qml.CNOT(wires=[wires[0], wires[1]])
    qml.U3(params[9], params[10], params[11], wires=wires[0])
    qml.U3(params[12], params[13], params[14], wires=wires[1])


def custom_Pooling1(params, wires):
    qml.Hadamard(wires=wires[0])
    qml.RZ(params[0], wires=wires[0])
    qml.Hadamard(wires=wires[1])
    qml.RZ(params[1], wires=wires[1])
    qml.Hadamard(wires=wires[2])
    qml.RZ(params[2], wires=wires[2])
    qml.Hadamard(wires=wires[3])
    qml.RZ(params[3], wires=wires[3])
    qml.IsingZZ(params[4], wires=[wires[0], wires[1]])
    qml.IsingZZ(params[5], wires=[wires[0], wires[2]])
    qml.IsingZZ(params[6], wires=[wires[0], wires[3]])
    qml.IsingZZ(params[7], wires=[wires[1], wires[2]])
    qml.IsingZZ(params[8], wires=[wires[1], wires[3]])
    qml.IsingZZ(params[9], wires=[wires[2], wires[3]])
    qml.CY(wires=[wires[0], wires[1]])
    qml.CY(wires=[wires[0], wires[2]])
    qml.CY(wires=[wires[0], wires[3]])
    qml.CY(wires=[wires[1], wires[2]])
    qml.CY(wires=[wires[1], wires[3]])
    qml.CY(wires=[wires[2], wires[3]])


def custom_Pooling4(params, wires):
    qml.Hadamard(wires=wires[0])
    qml.RZ(params[0], wires=wires[0])
    qml.Hadamard(wires=wires[1])
    qml.RZ(params[1], wires=wires[1])
    qml.Hadamard(wires=wires[2])
    qml.RZ(params[2], wires=wires[2])
    qml.Hadamard(wires=wires[3])
    qml.RZ(params[3], wires=wires[3])
    qml.IsingZZ(params[4], wires=[wires[0], wires[1]])
    qml.IsingZZ(params[5], wires=[wires[0], wires[2]])
    qml.IsingZZ(params[6], wires=[wires[0], wires[3]])
    qml.IsingZZ(params[7], wires=[wires[1], wires[2]])
    qml.IsingZZ(params[8], wires=[wires[1], wires[3]])
    qml.IsingZZ(params[9], wires=[wires[2], wires[3]])
    qml.RX(params[10], wires=wires[0])
    qml.RX(params[11], wires=wires[1])
    qml.RX(params[12], wires=wires[2])
    qml.RX(params[13], wires=wires[3])
    qml.CNOT(wires=[wires[0], wires[1]])
    qml.CNOT(wires=[wires[1], wires[2]])
    qml.CNOT(wires=[wires[2], wires[3]])
    qml.CNOT(wires=[wires[3], wires[0]])


def Pooling_ansatz1(params, wires):
    qml.CRZ(params[0], wires=[wires[0], wires[1]])
    qml.PauliX(wires=wires[0])
    qml.CRX(params[1], wires=[wires[0], wires[1]])


def Pooling_ansatz2(wires):
    qml.CRZ(wires=[wires[0], wires[1]])


def Pooling_ansatz3(*params, wires):
    qml.CRot(*params, wires=[wires[0], wires[1]])

# Embedding
---

In [4]:
def data_embedding(X, embedding_type='Amplitude'):
    if torch.is_tensor(X):
        X = X.detach().cpu().numpy()

    n_qubits = 8

    if embedding_type == 'Amplitude':
        num_features = len(X)
        if num_features > 2 ** n_qubits:
            raise ValueError(f"Input size ({num_features}) exceeds {n_qubits}-qubit capacity.")
        norm = np.linalg.norm(X)
        if norm == 0:
            raise ValueError("Input vector has zero norm.")
        normalized_X = X / norm
        padding = [0.0] * (2 ** n_qubits - num_features)
        amplitude_data = np.concatenate((normalized_X, padding))
        qml.AmplitudeEmbedding(amplitude_data, wires=range(n_qubits), normalize=False)

    elif embedding_type == 'Angle':
        for i in range(min(len(X), n_qubits)):
            qml.RX(X[i], wires=i)

    elif embedding_type == 'Angle-compact':
        num_pairs = len(X) // 2
        for i in range(num_pairs):
            qml.RY(X[2 * i], wires=i)
            qml.RZ(X[2 * i + 1], wires=i)

    elif embedding_type.startswith('Amplitude-Hybrid4-'):
        block = int(embedding_type.split('-')[-1])
        chunk_size = 2 ** 4
        start = (block - 1) * chunk_size
        X_chunk = X[start:start + chunk_size]
        if len(X_chunk) > 0:
            norm = np.linalg.norm(X_chunk)
            normalized = X_chunk / norm
            padding = [0.0] * (chunk_size - len(normalized))
            qml.AmplitudeEmbedding(np.concatenate((normalized, padding)),
                                   wires=range((block - 1) * 4, block * 4), normalize=False)

    elif embedding_type.startswith('Amplitude-Hybrid2-'):
        block = int(embedding_type.split('-')[-1])
        chunk_size = 2 ** 2
        start = (block - 1) * chunk_size
        X_chunk = X[start:start + chunk_size]
        if len(X_chunk) > 0:
            norm = np.linalg.norm(X_chunk)
            normalized = X_chunk / norm
            padding = [0.0] * (chunk_size - len(normalized))
            qml.AmplitudeEmbedding(np.concatenate((normalized, padding)),
                                   wires=range((block - 1) * 2, block * 2), normalize=False)

    elif embedding_type.startswith('Angular-Hybrid4-'):
        block = int(embedding_type.split('-')[-1])
        start = (block - 1) * 4
        for i, val in enumerate(X[start:start + 4]):
            qml.RX(val, wires=start + i)

    elif embedding_type.startswith('Angular-Hybrid2-'):
        block = int(embedding_type.split('-')[-1])
        start = (block - 1) * 2
        for i, val in enumerate(X[start:start + 2]):
            qml.RX(val, wires=start + i)

    else:
        raise ValueError(f"Unknown embedding type: {embedding_type}")

# QCNN Circuit
---

In [5]:
def conv_layer1(U, params):
    dispatch = {
        'U_TTN': U_TTN, 'U_5': U_5, 'U_6': U_6, 'U_9': U_9,
        'U_13': U_13, 'U_14': U_14, 'U_15': U_15, 'U_SO4': U_SO4, 'U_SU4': U_SU4,
        'U_SU4_no_pooling': U_SU4, 'U_SU4_1D': U_SU4, 'U_9_1D': U_9,
    }
    fn = dispatch.get(U)
    if fn is None:
        raise ValueError(f"Invalid unitary: {U}")
    for i in range(0, 8, 2):
        fn(params, wires=[i, i + 1])
    for i in range(1, 7, 2):
        fn(params, wires=[i, i + 1])
    fn(params, wires=[7, 0])


def conv_layer2(U, params):
    dispatch = {
        'U_TTN': U_TTN, 'U_5': U_5, 'U_6': U_6, 'U_9': U_9,
        'U_13': U_13, 'U_14': U_14, 'U_15': U_15, 'U_SO4': U_SO4, 'U_SU4': U_SU4,
        'U_SU4_no_pooling': U_SU4, 'U_SU4_1D': U_SU4, 'U_9_1D': U_9,
    }
    fn = dispatch.get(U)
    if fn is None:
        raise ValueError(f"Invalid unitary: {U}")
    for a, b in [(0, 2), (2, 4), (4, 6), (6, 0)]:
        fn(params, wires=[a, b])


def conv_layer3(U, params):
    dispatch = {
        'U_TTN': U_TTN, 'U_5': U_5, 'U_6': U_6, 'U_9': U_9,
        'U_13': U_13, 'U_14': U_14, 'U_15': U_15, 'U_SO4': U_SO4, 'U_SU4': U_SU4,
        'U_SU4_no_pooling': U_SU4, 'U_SU4_1D': U_SU4, 'U_9_1D': U_9,
    }
    fn = dispatch.get(U)
    if fn is None:
        raise ValueError(f"Invalid unitary: {U}")
    fn(params, wires=[0, 4])


def pooling_layer1(V, params):
    if V == 'Pooling_ansatz1':
        for i in range(0, 8, 2):
            Pooling_ansatz1(params, wires=[i + 1, i])
    elif V == 'Pooling_ansatz2':
        for i in range(0, 8, 2):
            Pooling_ansatz2(wires=[i + 1, i])
    elif V == 'Pooling_ansatz3':
        for i in range(0, 8, 2):
            Pooling_ansatz3(*params, wires=[i + 1, i])
    elif V == 'custom_Pooling4':
        custom_Pooling4(params, wires=[0, 1, 2, 3])
        custom_Pooling4(params, wires=[4, 5, 6, 7])
    elif V == 'custom_Pooling1':
        custom_Pooling1(params, wires=[0, 1, 2, 3])
        custom_Pooling1(params, wires=[4, 5, 6, 7])
    else:
        raise ValueError(f"Invalid pooling ansatz: {V}")


def pooling_layer2(V, params):
    if V == 'Pooling_ansatz1':
        Pooling_ansatz1(params, wires=[2, 0])
        Pooling_ansatz1(params, wires=[6, 4])
    elif V == 'Pooling_ansatz2':
        Pooling_ansatz2(wires=[2, 0])
        Pooling_ansatz2(wires=[6, 4])
    elif V == 'Pooling_ansatz3':
        Pooling_ansatz3(*params, wires=[2, 0])
        Pooling_ansatz3(*params, wires=[6, 4])
    elif V == 'custom_Pooling4':
        custom_Pooling4(params, wires=[0, 2, 4, 6])
    elif V == 'custom_Pooling1':
        custom_Pooling1(params, wires=[0, 2, 4, 6])
    else:
        raise ValueError(f"Invalid pooling ansatz: {V}")


def pooling_layer3(V, params):
    if V == 'Pooling_ansatz1':
        Pooling_ansatz1(params, wires=[4, 0])
    elif V == 'Pooling_ansatz2':
        Pooling_ansatz2(wires=[4, 0])
    elif V == 'Pooling_ansatz3':
        Pooling_ansatz3(*params, wires=[4, 0])
    else:
        raise ValueError(f"Invalid pooling ansatz: {V}")


def QCNN_structure(U, params, U_params):
    p1 = params[0:U_params[0]]
    p2 = params[U_params[0]:U_params[0] + U_params[2]]
    p3 = params[U_params[0] + U_params[2]:U_params[0] + U_params[2] + U_params[4]]
    p4 = params[U_params[0] + U_params[2] + U_params[4]:U_params[0] + U_params[2] + U_params[4] + U_params[1]]
    p5 = params[U_params[0] + U_params[2] + U_params[4] + U_params[1]:
              U_params[0] + U_params[2] + U_params[4] + U_params[1] + U_params[3]]
    p6 = params[U_params[0] + U_params[2] + U_params[4] + U_params[1] + U_params[3]:
              U_params[0] + U_params[2] + U_params[4] + U_params[1] + U_params[3] + U_params[5]]
    conv_layer1(U[0], p1)
    pooling_layer1(U[1], p4)
    conv_layer2(U[2], p2)
    pooling_layer2(U[3], p5)
    conv_layer3(U[4], p3)
    pooling_layer3(U[5], p6)


def QCNN_structure_without_pooling(U, params, U_params):
    p1 = params[0:U_params[0]]
    p2 = params[U_params[0]:U_params[0] + U_params[2]]
    p3 = params[U_params[0] + U_params[2]:U_params[0] + U_params[2] + U_params[4]]
    conv_layer1(U[0], p1)
    conv_layer2(U[2], p2)
    conv_layer3(U[4], p3)


def QCNN_1D_circuit(U, params, U_params):
    p1 = params[0:U_params[0]]
    p2 = params[U_params[0]:U_params[0] + U_params[2]]
    p3 = params[U_params[0] + U_params[2]:U_params[0] + U_params[2] + U_params[4]]
    unitary = U_9 if U[0] == 'U_9_1D' else U_SU4
    for i in range(0, 8, 2):
        unitary(p1, wires=[i, i + 1])
    for i in range(1, 7, 2):
        unitary(p1, wires=[i, i + 1])
    unitary(p2, wires=[2, 3])
    unitary(p2, wires=[4, 5])
    unitary(p3, wires=[3, 4])


dev = qml.device('default.qubit', wires=8)


@qml.qnode(dev, interface='torch', diff_method='backprop')
def QCNN(X, params, U, U_params, embedding_type='Amplitude', cost_fn='cross_entropy'):
    data_embedding(X, embedding_type=embedding_type)

    if 'U_SU4_no_pooling' in [U[0], U[2], U[4]]:
        QCNN_structure_without_pooling(U, params, U_params)
    elif 'U_SU4_1D' in [U[0], U[2], U[4]] or 'U_9_1D' in [U[0], U[2], U[4]]:
        QCNN_1D_circuit(U, params, U_params)
    else:
        QCNN_structure(U, params, U_params)

    if cost_fn == 'mse':
        return qml.expval(qml.PauliZ(4))
    return qml.probs(wires=4)

# PyTorch Model & Training
---

In [6]:
class TorchQCNN(nn.Module):
    """Wraps the PennyLane QCNN QNode as a trainable PyTorch module."""

    def __init__(self, U, U_params, embedding_type='Amplitude', cost_fn='cross_entropy'):
        super().__init__()
        total = sum(int(p) for p in U_params if p != '')
        self.weights = nn.Parameter(torch.randn(total) * 0.01)
        self.U = U
        self.U_params = U_params
        self.embedding_type = embedding_type
        self.cost_fn = cost_fn

    def forward(self, x):
        outputs = []
        for sample in x:
            out = QCNN(sample, self.weights, self.U, self.U_params,
                       self.embedding_type, self.cost_fn)
            outputs.append(out)
        return torch.stack(outputs)


def circuit_training(X_train, Y_train, U, U_params, embedding_type, circuit,
                     cost_fn, steps=200, learning_rate=0.01, batch_size=25):
    X_tensor = torch.tensor(X_train, dtype=torch.float32)

    if cost_fn == 'cross_entropy':
        Y_tensor = torch.tensor(Y_train, dtype=torch.long)
        criterion = nn.CrossEntropyLoss()
    else:
        Y_tensor = torch.tensor(Y_train, dtype=torch.float32)
        criterion = nn.MSELoss()

    model = TorchQCNN(U, U_params, embedding_type, cost_fn)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    loss_history = []
    n = len(X_train)

    for it in range(steps):
        idx = np.random.randint(0, n, (batch_size,))
        X_batch = X_tensor[idx]
        Y_batch = Y_tensor[idx]

        optimizer.zero_grad()
        preds = model(X_batch)

        if cost_fn == 'cross_entropy':
            loss = criterion(preds, Y_batch)
        else:
            loss = criterion(preds.squeeze(), Y_batch)

        loss.backward()
        optimizer.step()
        loss_history.append(loss.item())

        if it % 10 == 0:
            print(f"iteration: {it}  cost: {loss.item()}")

    return loss_history, model.weights.detach().cpu().numpy()


def predict_batch(X, trained_params, U, U_params, embedding_type, cost_fn):
    """Run inference on a list/array of samples."""
    params = torch.tensor(trained_params, dtype=torch.float32)
    X_tensor = torch.tensor(X, dtype=torch.float32)
    model = TorchQCNN(U, U_params, embedding_type, cost_fn)
    model.weights.data = params
    model.eval()
    with torch.no_grad():
        return model(X_tensor).numpy()

# Benchmarking
---

In [7]:
def accuracy_test(predictions, labels, cost_fn, binary=True):
    if cost_fn == 'mse':
        threshold = 1 if binary else 0.5
        acc = sum(1 for l, p in zip(labels, predictions) if abs(l - p) < threshold)
        return acc / len(labels)

    acc = 0
    for l, p in zip(labels, predictions):
        pred_class = 0 if p[0] > p[1] else 1
        if pred_class == l:
            acc += 1
    return acc / len(labels)


def Encoding_to_Embedding(Encoding):
    mapping = {
        'resize256': 'Amplitude',
        'pca8': 'Angle',
        'autoencoder8': 'Angle',
        'pca80-1': 'Amplitude-Hybrid4-1', 'autoencoder80-1': 'Amplitude-Hybrid4-1',
        'pca80-2': 'Amplitude-Hybrid4-2', 'autoencoder80-2': 'Amplitude-Hybrid4-2',
        'pca80-3': 'Amplitude-Hybrid4-3', 'autoencoder80-3': 'Amplitude-Hybrid4-3',
        'pca80-4': 'Amplitude-Hybrid4-4', 'autoencoder80-4': 'Amplitude-Hybrid4-4',
        'pca60-1': 'Amplitude-Hybrid4-1', 'autoencoder60-1': 'Amplitude-Hybrid4-1',
        'pca60-2': 'Amplitude-Hybrid4-2', 'autoencoder60-2': 'Amplitude-Hybrid4-2',
        'pca60-3': 'Amplitude-Hybrid4-3', 'autoencoder60-3': 'Amplitude-Hybrid4-3',
        'pca60-4': 'Amplitude-Hybrid4-4', 'autoencoder60-4': 'Amplitude-Hybrid4-4',
        'pca32-1': 'Amplitude-Hybrid4-1', 'autoencoder32-1': 'Amplitude-Hybrid4-1',
        'pca32-2': 'Amplitude-Hybrid4-2', 'autoencoder32-2': 'Amplitude-Hybrid4-2',
        'pca32-3': 'Amplitude-Hybrid4-3', 'autoencoder32-3': 'Amplitude-Hybrid4-3',
        'pca32-4': 'Amplitude-Hybrid4-4', 'autoencoder32-4': 'Amplitude-Hybrid4-4',
        'pca16-1': 'Amplitude-Hybrid2-1', 'autoencoder16-1': 'Amplitude-Hybrid2-1',
        'pca16-2': 'Amplitude-Hybrid2-2', 'autoencoder16-2': 'Amplitude-Hybrid2-2',
        'pca16-3': 'Amplitude-Hybrid2-3', 'autoencoder16-3': 'Amplitude-Hybrid2-3',
        'pca16-4': 'Amplitude-Hybrid2-4', 'autoencoder16-4': 'Amplitude-Hybrid2-4',
        'pca30-1': 'Angular-Hybrid4-1', 'autoencoder30-1': 'Angular-Hybrid4-1',
        'pca30-2': 'Angular-Hybrid4-2', 'autoencoder30-2': 'Angular-Hybrid4-2',
        'pca30-3': 'Angular-Hybrid4-3', 'autoencoder30-3': 'Angular-Hybrid4-3',
        'pca30-4': 'Angular-Hybrid4-4', 'autoencoder30-4': 'Angular-Hybrid4-4',
        'pca12-1': 'Angular-Hybrid2-1', 'autoencoder12-1': 'Angular-Hybrid2-1',
        'pca12-2': 'Angular-Hybrid2-2', 'autoencoder12-2': 'Angular-Hybrid2-2',
        'pca12-3': 'Angular-Hybrid2-3', 'autoencoder12-3': 'Angular-Hybrid2-3',
        'pca12-4': 'Angular-Hybrid2-4', 'autoencoder12-4': 'Angular-Hybrid2-4',
        'pca16-compact': 'Angle-compact', 'autoencoder16-compact': 'Angle-compact',
    }
    return mapping[Encoding]


def Benchmarking(dataset, classes, Unitaries, U_num_params, Encodings, circuit,
                 cost_fn, binary=True, result_file='result_mnist_pytorch.txt'):
    for U, U_params in zip(Unitaries, U_num_params):
        for Encoding in Encodings:
            Embedding = Encoding_to_Embedding(Encoding)
            X_train, X_test, Y_train, Y_test = data_load_and_process(
                dataset, classes=classes, feature_reduction=Encoding, binary=binary)

            print(f"\nLoss History for {circuit}, {U} {Encoding} with {cost_fn} on {dataset}")
            loss_history, trained_params = circuit_training(
                X_train, Y_train, U, U_params, Embedding, circuit, cost_fn)

            predictions = predict_batch(X_test, trained_params, U, U_params, Embedding, cost_fn)
            accuracy = accuracy_test(predictions, Y_test, cost_fn, binary)
            print(f"Accuracy for {U} {Encoding} on {dataset}: {accuracy}")

            with open(result_file, 'a+') as f:
                f.write(f"Loss History for {circuit}, {U} {Encoding} with {cost_fn} on {dataset}\n")
                f.write(str(loss_history) + "\n")
                f.write(f"Accuracy for {U} {Encoding} on {dataset}: {accuracy}\n\n")

# Run Benchmark
---

Same configuration as the original TensorFlow notebook.

In [8]:
Conv_Pooling_Layer_Ansatzs = [['U_SU4', 'custom_Pooling1', 'U_SU4', 'custom_Pooling4', 'U_SU4', 'Pooling_ansatz1']]
Conv_Pooling_Layer_Params = [[15, 10, 15, 14, 15, 2]]
Encodings = ['resize256', 'pca8', 'autoencoder8']
dataset = 'mnist'
classes = [0, 1]
binary = False
cost_fn = 'cross_entropy'

Benchmarking(
    dataset, classes, Conv_Pooling_Layer_Ansatzs, Conv_Pooling_Layer_Params,
    Encodings, circuit='QCNN', cost_fn=cost_fn, binary=binary,
)

Loaded X_train shape: (60000, 28, 28), Y_train shape: (60000,)
Loaded X_test shape: (10000, 28, 28), Y_test shape: (10000,)

Loss History for QCNN, ['U_SU4', 'custom_Pooling1', 'U_SU4', 'custom_Pooling4', 'U_SU4', 'Pooling_ansatz1'] resize256 with cross_entropy on mnist
iteration: 0  cost: 0.6884117250767615
iteration: 10  cost: 0.6233155049247001
iteration: 20  cost: 0.5608519979558824
iteration: 30  cost: 0.5415889369070134
iteration: 40  cost: 0.5495094687662524
iteration: 50  cost: 0.531309453681774
iteration: 60  cost: 0.5101823596698544
iteration: 70  cost: 0.5454093079561028
iteration: 80  cost: 0.4911961362990804
iteration: 90  cost: 0.4670946271705936
iteration: 100  cost: 0.48706442114245296
iteration: 110  cost: 0.4933472814176103
iteration: 120  cost: 0.4857835068621965
iteration: 130  cost: 0.497717286285391
iteration: 140  cost: 0.4874961010357785
iteration: 150  cost: 0.4654337049647428
iteration: 160  cost: 0.4950508747916611
iteration: 170  cost: 0.49462269686600435
it